In [ ]:
import json

import numpy as np
import plotly.graph_objects as go
import scipy.io as sio
from scipy.ndimage import map_coordinates

data = sio.loadmat("fields_002000.mat", squeeze_me=True)
# data = json.load(open("fields_002000.json", "r"))


def entry(container, key):
    """container[key] of a dict (JSON) or of a MATLAB structure read by loadmat; None if absent."""
    names = container.dtype.names if isinstance(container, np.ndarray) else container
    if names is None or key not in names:
        return None
    value = container[key]
    return value.item() if isinstance(value, np.ndarray) and value.dtype == object and value.ndim == 0 else value


nx = np.asarray(data['nx'], dtype=int)
lx = np.asarray(data['lx'], dtype=float)
if 'phi_a' in data.keys():
    phi_a = data['phi_a']
elif 'phi_A' in data.keys():
    phi_a = data['phi_A']
elif 'phi_A_real' in data.keys():
    phi_a = data['phi_A_real']      # CL-FTS: the real part
elif 'phi' in data.keys():
    phi_a = entry(data['phi'], 'A')
values = np.reshape(phi_a, nx)

# Lattice angles [alpha, beta, gamma] in degrees: saved by SCFT, in the checkpoint
# settings of L-FTS and CL-FTS fields files, or in the input of older files
if 'angles' in data.keys():
    angles = np.asarray(data['angles'], dtype=float)
elif 'checkpoint_settings_json' in data.keys():
    angles = np.asarray(json.loads(str(data['checkpoint_settings_json']))['angles'], dtype=float)
elif 'initial_params' in data.keys() and entry(data['initial_params'], 'angles') is not None:
    angles = np.asarray(entry(data['initial_params'], 'angles'), dtype=float)
else:
    angles = np.array([90.0, 90.0, 90.0])
    print("The file holds no lattice angles: the cell is drawn orthogonal.")
alpha, beta, gamma = np.deg2rad(angles)

# Lattice vectors as rows (a along x, b in the xy plane, as in ComputationBox)
cx = np.cos(beta)
cy = (np.cos(alpha) - np.cos(beta)*np.cos(gamma))/np.sin(gamma)
L = np.array([lx[0]*np.array([1.0, 0.0, 0.0]),
              lx[1]*np.array([np.cos(gamma), np.sin(gamma), 0.0]),
              lx[2]*np.array([cx, cy, np.sqrt(1.0 - cx**2 - cy**2)])])

# Fractional coordinates of the grid points, (i + offset)/n: the cell centres, or
# the nodes along a and b for a hexagonal-axes space group (Hall numbers 430-488
# except the rhombohedral-axes settings)
offsets = np.array([0.5, 0.5, 0.5])
hall = int(data['space_group_hall_number']) if 'space_group_hall_number' in data.keys() else 0
if 430 <= hall <= 488 and hall not in (434, 437, 445, 451, 453, 459, 461):
    offsets[:2] = 0.0

if np.allclose(angles, 90.0):
    # Orthogonal cell: the grid points themselves
    X, Y, Z = np.meshgrid(*[(np.arange(n) + o)/n*l for n, o, l in zip(nx, offsets, lx)], indexing='ij')
else:
    # Non-orthogonal cell: an isosurface needs points on a grid along x, y and z, so the
    # periodic field is interpolated linearly onto such a grid; it fills the box around
    # the cell, and the outline below marks the cell. The spacing is the finest distance
    # between the planes of the grid, widened when the box holds over 8 times the points
    corners = np.array(list(np.ndindex(2, 2, 2))) @ L
    low, high = corners.min(0), corners.max(0)
    h = np.min(1.0/(nx*np.linalg.norm(np.linalg.inv(L), axis=0)))
    n_box = np.prod((high - low)/h + 1)
    if n_box > 8*np.prod(nx):
        h *= (n_box/(8*np.prod(nx)))**(1/3)
        print(f"The box around the cell is large: drawn with spacing {h:.3g}.")
    X, Y, Z = np.meshgrid(*[np.arange(lo, hi + h/2, h) for lo, hi in zip(low, high)], indexing='ij')
    fractions = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1) @ np.linalg.inv(L)
    values = map_coordinates(values, (fractions*nx - offsets).T, order=1, mode='grid-wrap').reshape(X.shape)

# The edges of the cell
edges = []
for corner in np.ndindex(2, 2, 2):
    for d in range(3):
        if corner[d] == 0:
            end = list(corner)
            end[d] = 1
            edges += [np.array(corner) @ L, np.array(end) @ L, [None]*3]
edges = np.array(edges, dtype=object)

fig = go.Figure(data=[go.Isosurface(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=values.flatten(),
    # isomax=1.0,
    # opacity=1.0,
    # surface_count=2, # number of isosurface. 2 by default:only min and max
    isomin=0.0,
    isomax=1.0,
    opacity=0.2,
    surface_count=9, # number of isosurface. 2 by default:only min and max
    #caps=dict(x_show=False, y_show=False, z_show=False),
    colorscale='RdBu', #RdBu, jet, Plotly3, OrRd
    colorbar=dict(thickness=30,tickfont=dict(size=30,color="black")),
    reversescale=True,
    ),
    go.Scatter3d(x=edges[:, 0], y=edges[:, 1], z=edges[:, 2], mode='lines',
                 line=dict(color='black', width=2), showlegend=False)])
fig.update_layout(
    autosize=False,
    width=500,
    height=500,
)
fig.update_scenes(
    aspectmode="data",
    camera_projection_type="orthographic",
    xaxis_visible=False,
    yaxis_visible=False,
    zaxis_visible=False)

# fig.write_html('3d_isosurface.html', auto_open=True)
# fig.write_image('3d_volume.svg')